# Task 1 — Exploratory Data Analysis
**ACIS Insurance Risk Analytics**

Goals:
1. Summarise the dataset and check quality.
2. Compute the portfolio **Loss Ratio** and slice it by Province / VehicleType / Gender.
3. Surface the distributions, outliers, temporal trends, and geographic patterns.
4. Produce 3+ insight-driven plots.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_insurance_data, add_derived_metrics
from src import eda_utils as eu

sns.set_theme(style='whitegrid')
pd.options.display.max_columns = 80

In [ ]:
DATA_PATH = '../data/insurance_data.csv'  # update to match the downloaded file
df = load_insurance_data(DATA_PATH)
df = add_derived_metrics(df)
print(df.shape)
df.head()

## 1. Data summary & dtype audit

In [ ]:
df.dtypes.value_counts()

In [ ]:
eu.numeric_summary(df)

## 2. Missing values

In [ ]:
eu.missing_value_report(df)

**Handling strategy** (document decisions here):
- High-missing columns (>50%): drop or treat as 'unknown' category.
- Categorical: impute with mode or `Unknown`.
- Numeric: impute with median (preserves skewed insurance distributions).
- Always handle inside a pipeline (see `src/modeling.build_preprocessor`).

## 3. Portfolio loss ratio

In [ ]:
portfolio_loss_ratio = df['TotalClaims'].sum() / df['TotalPremium'].sum()
print(f'Portfolio Loss Ratio: {portfolio_loss_ratio:.3f}')

In [ ]:
for col in ['Province', 'VehicleType', 'Gender']:
    if col in df.columns:
        display(eu.loss_ratio_by(df, col).head(10))

## 4. Univariate distributions

In [ ]:
eu.plot_numeric_distributions(
    df,
    ['TotalPremium', 'TotalClaims', 'SumInsured', 'CustomValueEstimate',
     'CalculatedPremiumPerTerm', 'Kilowatts']
)
plt.show()

In [ ]:
eu.plot_categorical_counts(df, ['Province', 'Gender', 'CoverType', 'VehicleType'])
plt.show()

## 5. Outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['TotalClaims', 'TotalPremium', 'CustomValueEstimate']):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(col)
fig.tight_layout()

## 6. Geographic patterns (Insight Plot 1)

In [ ]:
eu.plot_loss_ratio_by(df, 'Province')
plt.show()

## 7. Temporal trends (Insight Plot 2)

In [ ]:
fig, monthly = eu.plot_temporal_trends(df)
plt.show()
monthly.tail()

## 8. Vehicle make/model vs claim amount (Insight Plot 3)

In [ ]:
claim_by_make = (
    df[df['HasClaim'] == 1]
    .groupby('Make', observed=True)['TotalClaims']
    .agg(['mean', 'count'])
    .query('count >= 30')
    .sort_values('mean', ascending=False)
)
top = claim_by_make.head(10)
bottom = claim_by_make.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x=top['mean'], y=top.index.astype(str), ax=axes[0], color='#c44536')
axes[0].set_title('Top 10 makes by mean claim amount')
sns.barplot(x=bottom['mean'], y=bottom.index.astype(str), ax=axes[1], color='#3a7ca5')
axes[1].set_title('Bottom 10 makes by mean claim amount')
fig.tight_layout()

## 9. Premium vs Claims correlation by PostalCode

In [ ]:
if 'PostalCode' in df.columns:
    by_zip = df.groupby('PostalCode', observed=True).agg(
        premium=('TotalPremium', 'sum'),
        claims=('TotalClaims', 'sum'),
        policies=('TotalPremium', 'size'),
    ).query('policies >= 30')
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(data=by_zip, x='premium', y='claims', size='policies',
                    sizes=(20, 300), alpha=0.6, ax=ax, legend=False)
    ax.plot([0, by_zip['premium'].max()], [0, by_zip['premium'].max()],
            'k--', linewidth=1, label='Break-even (claims = premium)')
    ax.set_xlabel('Total Premium (Rand)')
    ax.set_ylabel('Total Claims (Rand)')
    ax.set_title('Premium vs Claims by Postal Code')
    ax.legend()
    plt.show()

## Findings (fill in after running)
- Portfolio loss ratio: …
- Highest-risk provinces: …
- Most/least claim-prone vehicle makes: …
- Temporal trends: …
- Notable outliers and how we'll handle them: …